# AAV9 fitting protocol -- cross-packaging vs NGS hallucination, impact comparison

Trimmed-down descendant of `AAV9_fitting_protocol.ipynb`: keeps its Part 0-2b setup (real
aav9.csv data, `F_viab`/`J_viab` ground-truth weights, deterministic `viab_score_GT`, `R_REAL`)
and its Part 5/6 "manual playground" comparison (simulated `target1` vs real `target`, and vs
`viab_score_GT`) -- but drops Part 1 (5-run repeatability check) and Part 2 (random
hyperparameter search over `mu`/`T_viab`/`noise_viab`/`D`), which aren't needed here. Part 7
(ProfileMLP trained on real `target`) is also dropped -- it never touches simulated `target1`,
so it can't show anything about these two noise sources' impact.

Runs the SAME baseline hyperparameters (`mu=50`, `T_viab=0.8`, `noise_viab=0.5`, `D=1e9` --
this notebook's own established "best realistic" operating point) through 4 `Protocol`
variants from `lib/cross_packaging_draft.py`:

1. **baseline** -- plain `ProtocolV3`, no extra noise source
2. **cross-packaging** -- `ProtocolCrossPackagingBackground`: non-viable variants leak a
   floor of capsid production from co-transfected neighbors (see the module docstring)
3. **hallucination** -- `ProtocolWithHallucination`: a small fraction of sequences get
   spurious NGS reads at each checkpoint, independent of their true abundance (see
   `new_variant_appearance_analysis.ipynb`)
4. **both** -- `ProtocolCrossPackagingAndHallucination`, both sources combined

to see how each distorts the simulated log-enrichment distribution relative to the real
`aav9.csv` target and to the deterministic GT score.

## 0. Setup

In [ ]:
import sys, os

# This notebook lives in "Modelization_V1/notebooks/aav_viability_test/", two levels below
# analysisV1.py / sequence_classesV1.py -- same sys.path convention as AAV9_fitting_protocol.ipynb.
sys.path.insert(0, os.path.abspath("../.."))
sys.path.insert(0, os.path.abspath("."))
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # must be set before jax initializes

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

from analysisV1 import AA_LABELS, pearson
from sequence_classesV1 import ProtocolV3
from cross_packaging_draft import (
    ProtocolCrossPackagingBackground, ProtocolWithHallucination,
    ProtocolCrossPackagingAndHallucination,
)
from initialize_weights import (
# GT swapped 2026-08-27: now loads the joint Potts-regression GT (AAV9_potts_regression.ipynb) instead of the naive group-means one -- cell outputs below were cleared since they were computed under the old GT; re-run this notebook before trusting any number in it.
    load_F_viab_aav9_potts, load_J_viab_aav9_potts, NUM_AMINO_ACIDS, NUM_POSITIONS,
)

print(f"JAX backend: {jax.default_backend()} -- devices: {jax.devices()}")

## 1. Real AAV9 data + ground-truth weights

In [ ]:
df = pd.read_csv("aav9.csv")
print(f"{len(df):,} variants, columns: {list(df.columns)}")

lengths = df["sequence"].str.len().unique()
assert len(lengths) == 1 and int(lengths[0]) == NUM_POSITIONS

lut = np.zeros(256, dtype=np.int64)
for i, aa in enumerate(AA_LABELS):
    lut[ord(aa)] = i

seq_bytes   = np.frombuffer("".join(df["sequence"]).encode("ascii"), dtype=np.uint8)
seq_matrix  = lut[seq_bytes].reshape(len(df), NUM_POSITIONS)
real_target = df["target"].to_numpy()

sequences = jnp.asarray(seq_matrix)
d0        = sequences.shape[0]

# GT swapped 2026-08-27: now loads the joint Potts-regression GT (AAV9_potts_regression.ipynb) instead of the naive group-means one -- cell outputs below were cleared since they were computed under the old GT; re-run this notebook before trusting any number in it.
F_viab = load_F_viab_aav9_potts()
J_viab = load_J_viab_aav9_potts()
F_sel  = jnp.zeros_like(F_viab)
J_sel  = jnp.zeros_like(J_viab)

print(f"d0 = {d0:,} real AAV9 library sequences")

## 2. Baseline config + deterministic GT score

Same baseline as `AAV9_fitting_protocol.ipynb` section 2 (`mu=50`, `T_viab=0.8`,
`noise_viab=0.5`, `D=1e9`) -- shared by all 4 variants below, so the ONLY thing that differs
between them is the extra noise source, not the underlying operating point.

`viability_target1()` and `R_REAL` are copied over unchanged (section 1's `viability_target1`,
section 4's `R_REAL` definition) -- both are cheap, neither needs the repeatability run or the
hyperparameter search that were dropped.

In [ ]:
RHO_REF          = 1e-3
MU_BASE          = 50
T_VIAB_BASE      = 0.8
NOISE_VIAB_BASE  = 0.5
D_BASE           = 1e9
DILUTION_FACTOR  = 10
eps              = 1.0

N1_BASE = MU_BASE * d0 / RHO_REF
N0_BASE = N1_BASE * 10

BASE_KWARGS = dict(
    multinomialNGS=True, N0=N0_BASE, N1=N1_BASE, dilution_factor=DILUTION_FACTOR,
    sequences=sequences, D=D_BASE, F_viab=F_viab, J_viab=J_viab, F_sel=F_sel, J_sel=J_sel,
    noise_viab=NOISE_VIAB_BASE, noise_sel=0.0, T_sel=1.0, T_viab=T_VIAB_BASE,
)

def viability_target1(protocol, eps=eps):
    """Runs one loop_DE() round and returns target1 = log((lambda2p+eps)/(lambda0p+eps)),
    same formula as AAV9_fitting_protocol.ipynb. Resets lambda0 first since loop_DE()
    depletes it in place."""
    protocol.lambda0 = jnp.full(protocol.d0, protocol.N0 / protocol.d0)
    _bio_row, ngs_row = protocol.loop_DE()
    lambda0p, lambda2p, _lambda3p = (np.asarray(a) for a in ngs_row)
    return np.log((lambda2p + eps) / (lambda0p + eps))

viab_score_GT = np.asarray(ProtocolV3(**BASE_KWARGS).compute_score(F_viab, J_viab))
R_REAL        = pearson(real_target, viab_score_GT)
print(f"R_REAL = r(real aav9 target, deterministic GT score) = {R_REAL:.3f}")

## 3. Four protocol variants, same baseline config

`CROSS_PACKAGING_RATE` is a free-standing illustration value (not empirically calibrated --
see `ProtocolCrossPackagingBackground`'s own docstring). `HALLUCINATION_RATE` defaults to the
empirically-measured rate from `new_variant_appearance_analysis.ipynb`
(69/74,464 candidate variants on `fit4functionaav9.csv`). Edit either, then re-run this cell
and everything below.

In [ ]:
CROSS_PACKAGING_RATE = 0.05          # fraction of the "healthy neighbor" median rate leaked to every sequence
HALLUCINATION_RATE    = 0.17  # empirical rate from new_variant_appearance_analysis.ipynb

variants = {
    "baseline":         ProtocolV3(**BASE_KWARGS),
    "cross-packaging":  ProtocolCrossPackagingBackground(**BASE_KWARGS, cross_packaging_rate=CROSS_PACKAGING_RATE),
    "hallucination":    ProtocolWithHallucination(**BASE_KWARGS, hallucination=True, hallucination_rate=HALLUCINATION_RATE),
    "both":             ProtocolCrossPackagingAndHallucination(**BASE_KWARGS, cross_packaging_rate=CROSS_PACKAGING_RATE,
                                                                 hallucination=True, hallucination_rate=HALLUCINATION_RATE),
}

results = {}
for name, protocol in variants.items():
    target1 = viability_target1(protocol)
    results[name] = dict(
        target1=target1,
        r_sim_real=pearson(target1, real_target),
        r_sim_gt=pearson(target1, viab_score_GT),
        frac_zero=float(np.mean(target1 == 0.0)),
    )
    print(f"{name:18s}  r(sim,real)={results[name]['r_sim_real']:.3f}  "
          f"r(sim,GT)={results[name]['r_sim_gt']:.3f}  frac(target1=0)={results[name]['frac_zero']:.3f}")

## 4. Simulated `target1` vs real `target` -- one panel per variant

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
bins_real = np.linspace(
    min(real_target.min(), *(r["target1"].min() for r in results.values())),
    max(real_target.max(), *(r["target1"].max() for r in results.values())),
    60,
)

for ax, (name, r) in zip(axes.ravel(), results.items()):
    ax.hist(real_target,   bins=bins_real, alpha=0.55, density=True, color="tab:orange", label="real aav9 target")
    ax.hist(r["target1"],  bins=bins_real, alpha=0.55, density=True, color="tab:blue",   label="simulated target1")
    ax.set_title(f"{name}\nr(sim,real)={r['r_sim_real']:.3f}  R_REAL={R_REAL:.3f}  frac(target1=0)={r['frac_zero']:.3f}",
                 fontsize=10)
    ax.set_xlabel("log enrichment")
    ax.set_ylabel("density")
    ax.legend(fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.3)

fig.suptitle(f"mu={MU_BASE}  T_viab={T_VIAB_BASE}  noise_viab={NOISE_VIAB_BASE}  D={D_BASE:.0e}  "
             f"(cross_packaging_rate={CROSS_PACKAGING_RATE}, hallucination_rate={HALLUCINATION_RATE:.6f} where applicable)")
fig.tight_layout()
plt.show()

## 5. Simulated `target1` vs deterministic GT score -- one panel per variant

Same idea as `AAV9_fitting_protocol.ipynb` section 6: `viab_score_GT` (no protocol noise at
all) against each variant's `target1`, to see how much of the shape distortion comes from the
`T_viab`/noise transform itself vs. from the two extra noise sources.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
bins_gt = np.linspace(
    min(viab_score_GT.min(), *(r["target1"].min() for r in results.values())),
    max(viab_score_GT.max(), *(r["target1"].max() for r in results.values())),
    60,
)

for ax, (name, r) in zip(axes.ravel(), results.items()):
    ax.hist(viab_score_GT, bins=bins_gt, alpha=0.55, density=True, color="tab:green", label="GT score (F+J, no noise)")
    ax.hist(r["target1"],  bins=bins_gt, alpha=0.55, density=True, color="tab:blue",  label="simulated target1")
    ax.set_title(f"{name}\nr(sim,GT)={r['r_sim_gt']:.3f}  R_REAL={R_REAL:.3f}", fontsize=10)
    ax.set_xlabel("value")
    ax.set_ylabel("density")
    ax.legend(fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.3)

fig.tight_layout()
plt.show()

## 6. Summary

In [ ]:
summary_df = pd.DataFrame({
    name: {"r(sim, real)": r["r_sim_real"], "r(sim, GT)": r["r_sim_gt"], "frac(target1=0)": r["frac_zero"]}
    for name, r in results.items()
}).T
summary_df.loc["R_REAL (reference)"] = [R_REAL, np.nan, np.nan]
summary_df